# 24CS2506 – Building Applications using GPT-4
## Experiment 1 – Building an AI-Powered E-Commerce Chatbot using Ollama

In [1]:
!pip install pandas numpy scikit-learn sentence-transformers transformers torch ollama

  Using cached sentence_transformers-6.0.1-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached regex-2026.9.3-cp314-cp314-win_amd64.whl.metadata (41 kB)
Using cached sentence_transformers-6.0.1-py3-none-any.whl (739 kB)
Using cached transformers-5.16.1-py3-none-any.whl (12.1 MB)
Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl (2.9 MB)
Using cached regex-2026.9.3-cp314-cp314-win_amd64.whl (280 kB)

   ---------------------------------------- 0/4 [regex]
   ---------- ----------------------------- 1/4 [tokenizers]
   -------------------- ------------------- 2/4 [transformers]
   -------------------- ------------------- 2/4 [transformers]
   -------------------- ------------------- 2/4 [transformers]
   -------------------- ------------------- 2/4 [transformers]
   -------------------- ------------------- 2/4 [transformers]
   ---------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import ollama

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline

C:\Users\rishi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os

print("Current Jupyter folder:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

Current Jupyter folder:
C:\Users\rishi

Files in this folder:
['.android', '.aws', '.azure', '.blip-device-dedupe-id', '.cache', '.cagent', '.chocolatey', '.claude', '.claude.json', '.claude.json.tmp.2432.31c0ed94dcbb', '.config', '.copilot', '.cursor', '.docker', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.kaggle', '.kube', '.lesshst', '.matplotlib', '.minikube', '.ollama', '.openclaw', '.skiko', '.viminfo', '.vscode', '.vscode-shared', '.zshrc', 'adaboost.ipynb', 'animation_app', 'AppData', 'Application Data', 'apriori.ipynb', 'B5_driverFCM.csv', 'B5_iris_regression.csv', 'B5_iris_SLP.csv', 'Contacts', 'Cookies', 'Documents', 'Downloads', 'Ex_9_MLT (2).ipynb', 'faqs.csv', 'Favorites', 'fcm.ipynb', 'iris_SLP.csv', 'linear_reg.ipynb', 'Links', 'Local Settings', 'logistic.ipynb', 'media_app', 'mlp.ipynb', 'multi_linear.ipynb', 'Music', 'My Documents', 'navigation_app', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{b42eb9aa-df75-11f0-a195-fc

In [6]:
import os

for root, dirs, files in os.walk(os.getcwd()):
    for file in files:
        if file in ["products.csv", "faqs.csv", "reviews.csv"]:
            print(os.path.join(root, file))

C:\Users\rishi\faqs.csv
C:\Users\rishi\products.csv
C:\Users\rishi\reviews.csv
C:\Users\rishi\Downloads\faqs.csv
C:\Users\rishi\Downloads\products.csv
C:\Users\rishi\Downloads\reviews.csv


In [7]:
import os

print("Jupyter is currently running from:")
print(os.getcwd())

print("\nSearching for your CSV files...\n")

for root, dirs, files in os.walk(os.getcwd()):
    for file in files:
        if file in ["products.csv", "faqs.csv", "reviews.csv"]:
            print(os.path.join(root, file))

Jupyter is currently running from:
C:\Users\rishi

Searching for your CSV files...

C:\Users\rishi\faqs.csv
C:\Users\rishi\products.csv
C:\Users\rishi\reviews.csv
C:\Users\rishi\Downloads\faqs.csv
C:\Users\rishi\Downloads\products.csv
C:\Users\rishi\Downloads\reviews.csv


In [9]:
import pandas as pd

products = pd.read_csv("products.csv")
faqs = pd.read_csv("faqs.csv")
reviews = pd.read_csv("reviews.csv")

print("Products:", products.shape)
print("FAQs:", faqs.shape)
print("Reviews:", reviews.shape)

Products: (100, 7)
FAQs: (20, 3)
Reviews: (1000, 5)


In [10]:
display(products.head())
display(faqs.head())
display(reviews.head())

,ProductID,ProductName,Category,Brand,Price,Description,Rating
0,1,Lenovo Laptop 1,Laptop,Lenovo,30316,Laptop by Lenovo suitable for everyday use wit...,4.5
1,2,Lenovo Laptop 2,Laptop,Lenovo,72664,Laptop by Lenovo suitable for everyday use wit...,3.8
2,3,Lenovo Laptop 3,Laptop,Lenovo,7081,Laptop by Lenovo suitable for everyday use wit...,4.3
3,4,Lenovo Laptop 4,Laptop,Lenovo,51158,Laptop by Lenovo suitable for everyday use wit...,3.8
4,5,Lenovo Laptop 5,Laptop,Lenovo,80105,Laptop by Lenovo suitable for everyday use wit...,3.9


,FAQID,Question,Answer
0,1,How do I track my order?,Use the Track Order page with your order ID.
1,2,Where is my parcel?,Use the Track Order page with your order ID.
2,3,How do I return my order?,Initiate a return within 7 days from My Orders.
3,4,Can I cancel my order?,Orders can be cancelled before shipment.
4,5,How long is delivery?,Delivery usually takes 3-7 business days.


,ReviewID,ProductID,ProductName,Review,Rating
0,1,1,Lenovo Laptop 1,Not worth the price,3
1,2,1,Lenovo Laptop 1,Works perfectly,5
2,3,1,Lenovo Laptop 1,Very good quality,4
3,4,1,Lenovo Laptop 1,Excellent product,5
4,5,1,Lenovo Laptop 1,Very good quality,5


In [11]:
products = pd.read_csv("products.csv")
faqs = pd.read_csv("faqs.csv")
reviews = pd.read_csv("reviews.csv")

print("Products:", products.shape)
print("FAQs:", faqs.shape)
print("Reviews:", reviews.shape)

Products: (100, 7)
FAQs: (20, 3)
Reviews: (1000, 5)


In [12]:
print("===== PRODUCTS =====")
display(products.head())

print("===== FAQs =====")
display(faqs.head())

print("===== REVIEWS =====")
display(reviews.head())

===== PRODUCTS =====


,ProductID,ProductName,Category,Brand,Price,Description,Rating
0,1,Lenovo Laptop 1,Laptop,Lenovo,30316,Laptop by Lenovo suitable for everyday use wit...,4.5
1,2,Lenovo Laptop 2,Laptop,Lenovo,72664,Laptop by Lenovo suitable for everyday use wit...,3.8
2,3,Lenovo Laptop 3,Laptop,Lenovo,7081,Laptop by Lenovo suitable for everyday use wit...,4.3
3,4,Lenovo Laptop 4,Laptop,Lenovo,51158,Laptop by Lenovo suitable for everyday use wit...,3.8
4,5,Lenovo Laptop 5,Laptop,Lenovo,80105,Laptop by Lenovo suitable for everyday use wit...,3.9


===== FAQs =====


,FAQID,Question,Answer
0,1,How do I track my order?,Use the Track Order page with your order ID.
1,2,Where is my parcel?,Use the Track Order page with your order ID.
2,3,How do I return my order?,Initiate a return within 7 days from My Orders.
3,4,Can I cancel my order?,Orders can be cancelled before shipment.
4,5,How long is delivery?,Delivery usually takes 3-7 business days.


===== REVIEWS =====


,ReviewID,ProductID,ProductName,Review,Rating
0,1,1,Lenovo Laptop 1,Not worth the price,3
1,2,1,Lenovo Laptop 1,Works perfectly,5
2,3,1,Lenovo Laptop 1,Very good quality,4
3,4,1,Lenovo Laptop 1,Excellent product,5
4,5,1,Lenovo Laptop 1,Very good quality,5


In [13]:
print("===== PRODUCTS =====")
print("Rows:", products.shape[0])
print("Columns:", products.shape[1])
print("Attributes:", list(products.columns))

print("\n===== FAQs =====")
print("Rows:", faqs.shape[0])
print("Columns:", faqs.shape[1])
print("Attributes:", list(faqs.columns))

print("\n===== REVIEWS =====")
print("Rows:", reviews.shape[0])
print("Columns:", reviews.shape[1])
print("Attributes:", list(reviews.columns))

===== PRODUCTS =====
Rows: 100
Columns: 7
Attributes: ['ProductID', 'ProductName', 'Category', 'Brand', 'Price', 'Description', 'Rating']

===== FAQs =====
Rows: 20
Columns: 3
Attributes: ['FAQID', 'Question', 'Answer']

===== REVIEWS =====
Rows: 1000
Columns: 5
Attributes: ['ReviewID', 'ProductID', 'ProductName', 'Review', 'Rating']


## Q2 – Dataset Description

### Products Dataset
Contains product information such as ProductID, ProductName, Category, Brand, Price, Description and Rating. It is used for product recommendation and product-specific question answering.

### FAQ Dataset
Contains frequently asked questions and their corresponding answers. It is used for semantic FAQ search and customer-support responses.

### Reviews Dataset
Contains customer reviews associated with products along with ratings. It is used for sentiment analysis, review summarization and buying suggestions.

In [14]:
faq_model = SentenceTransformer("all-MiniLM-L6-v2")

print("FAQ embedding model loaded successfully!")

C:\Users\rishi\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rishi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█████████████████████████████████████████████████████

FAQ embedding model loaded successfully!


In [15]:
faq_questions = faqs["Question"].fillna("").tolist()

faq_embeddings = faq_model.encode(
    faq_questions,
    convert_to_numpy=True
)

print("FAQ embeddings generated successfully!")
print("Number of embeddings:", len(faq_embeddings))
print("Embedding dimensions:", faq_embeddings.shape[1])

FAQ embeddings generated successfully!
Number of embeddings: 20
Embedding dimensions: 384


In [16]:
np.save("faq_embeddings.npy", faq_embeddings)

print("FAQ embeddings saved successfully!")

FAQ embeddings saved successfully!


In [17]:
def search_faq(question):
    
    query_embedding = faq_model.encode(
        [question],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        query_embedding,
        faq_embeddings
    )[0]

    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    best_question = faqs.iloc[best_index]["Question"]
    best_answer = faqs.iloc[best_index]["Answer"]

    return best_question, best_answer, best_score

In [18]:
question = "Where is my parcel?"

best_question, best_answer, score = search_faq(question)

print("User Question:", question)
print("Best Matching FAQ:", best_question)
print("Similarity Score:", round(score, 3))
print("Answer:", best_answer)

User Question: Where is my parcel?
Best Matching FAQ: Where is my parcel?
Similarity Score: 1.0
Answer: Use the Track Order page with your order ID.


In [19]:
FAQ_THRESHOLD = 0.50

def search_faq_with_threshold(question):
    
    best_question, best_answer, score = search_faq(question)

    print("Best Matching FAQ:", best_question)
    print("Similarity Score:", round(score, 3))

    if score >= FAQ_THRESHOLD:
        return best_question, best_answer, score

    return None, None, score

In [20]:
question = "Where is my parcel?"

faq_question, faq_answer, score = search_faq_with_threshold(question)

if faq_question:
    print("\nRelevant FAQ found!")
    print("Answer:", faq_answer)
else:
    print("\nNo sufficiently relevant FAQ found.")

Best Matching FAQ: Where is my parcel?
Similarity Score: 1.0

Relevant FAQ found!
Answer: Use the Track Order page with your order ID.


In [22]:
!ollama list

NAME            ID              SIZE      MODIFIED    
qwen2.5:0.5b    a8b0c5157701    397 MB    3 weeks ago    


In [24]:
import ollama

response = ollama.chat(
    model="qwen2.5:0.5b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response["message"]["content"])

Hello! I'm here to help you with any questions or topics you might have. Feel free to ask me anything you'd like!


In [26]:
def generate_faq_response(user_question, faq_question, faq_answer):

    prompt = f"""
You are an e-commerce customer support assistant.

User question:
{user_question}

Retrieved FAQ question:
{faq_question}

Retrieved FAQ answer:
{faq_answer}

Answer the user's question naturally.

Rules:
1. Use only the information in the retrieved FAQ answer.
2. Do not invent information.
3. Keep the response concise.
"""

    response = ollama.chat(
        model="qwen2.5:0.5b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [27]:
question = "Where is my parcel?"

faq_question, faq_answer, score = search_faq_with_threshold(question)

if faq_question:

    answer = generate_faq_response(
        question,
        faq_question,
        faq_answer
    )

    print("User:", question)
    print("\nBest Matching FAQ:", faq_question)
    print("Similarity Score:", round(score, 3))

    print("\nChatbot:")
    print(answer)

else:
    print("No relevant FAQ found.")

Best Matching FAQ: Where is my parcel?
Similarity Score: 1.0
User: Where is my parcel?

Best Matching FAQ: Where is my parcel?
Similarity Score: 1.0

Chatbot:
To find your parcel, use the Track Order page with your order ID.


In [28]:
print(products.columns.tolist())

['ProductID', 'ProductName', 'Category', 'Brand', 'Price', 'Description', 'Rating']


In [29]:
products["Description"] = products["Description"].fillna("")

tfidf = TfidfVectorizer(
    stop_words="english"
)

product_vectors = tfidf.fit_transform(
    products["Description"]
)

print("TF-IDF vectors created successfully!")
print("Products:", product_vectors.shape[0])
print("Features:", product_vectors.shape[1])

TF-IDF vectors created successfully!
Products: 100
Features: 25


In [30]:
def retrieve_products(query, top_k=3):

    query_vector = tfidf.transform([query])

    similarities = cosine_similarity(
        query_vector,
        product_vectors
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = products.iloc[top_indices].copy()

    results["Similarity"] = similarities[top_indices]

    return results

In [31]:
query = "Recommend a gaming laptop"

results = retrieve_products(query)

print("Search Product:", query)
print("\nTop 3 Retrieved Products:")

display(
    results[
        [
            "ProductName",
            "Brand",
            "Category",
            "Price",
            "Rating",
            "Similarity"
        ]
    ]
)

Search Product: Recommend a gaming laptop

Top 3 Retrieved Products:


,ProductName,Brand,Category,Price,Rating,Similarity
12,Dell Laptop 3,Dell,Laptop,11870,4.3,0.469776
13,Dell Laptop 4,Dell,Laptop,56062,3.9,0.469776
14,Dell Laptop 5,Dell,Laptop,86595,3.8,0.469776


In [32]:
def generate_product_recommendation(query, results):

    context = ""

    for _, product in results.iterrows():

        context += f"""
Product Name: {product['ProductName']}
Brand: {product['Brand']}
Category: {product['Category']}
Price: ₹{product['Price']}
Description: {product['Description']}
Rating: {product['Rating']}
"""

    prompt = f"""
You are an e-commerce product recommendation assistant.

User request:
{query}

Retrieved products:
{context}

Select the single best product.

Give a short justification.

Rules:
- Choose only from the retrieved products.
- Mention the product name.
- Mention the brand.
- Mention the price.
- Use only the provided information.
- Do not invent specifications.
"""

    response = ollama.chat(
        model="qwen2.5:0.5b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [33]:
query = "Recommend a gaming laptop"

results = retrieve_products(query)

print("Retrieved Products:")
display(
    results[
        [
            "ProductName",
            "Brand",
            "Price",
            "Rating",
            "Similarity"
        ]
    ]
)

recommendation = generate_product_recommendation(
    query,
    results
)

print("\nOllama Recommendation:")
print(recommendation)

Retrieved Products:


,ProductName,Brand,Price,Rating,Similarity
12,Dell Laptop 3,Dell,11870,4.3,0.469776
13,Dell Laptop 4,Dell,56062,3.9,0.469776
14,Dell Laptop 5,Dell,86595,3.8,0.469776



Ollama Recommendation:
Dell Laptop 5 - Quality performance, suitable for everyday use with quality performance. Price: ₹86595. Rating: 3.8

Justification:
I have selected Dell Laptop 5 based on its high price and good rating of 3.8 out of 4.3. It is the best recommendation among the options given because it offers a combination of quality in terms of performance and reasonable prices, suitable for most everyday users.


In [34]:
def retrieve_product(query):

    query_vector = tfidf.transform([query])

    similarities = cosine_similarity(
        query_vector,
        product_vectors
    )[0]

    best_index = np.argmax(similarities)

    return (
        products.iloc[best_index],
        similarities[best_index]
    )
    

In [35]:
query = "Tell me about Lenovo Laptop 3"

product, similarity = retrieve_product(query)

print("Retrieved Product:", product["ProductName"])
print("Similarity:", round(similarity, 3))

Retrieved Product: Lenovo Laptop 1
Similarity: 0.895


In [36]:
def product_question_answer(query):

    product, similarity = retrieve_product(query)

    context = f"""
Product Name: {product['ProductName']}
Brand: {product['Brand']}
Category: {product['Category']}
Price: ₹{product['Price']}
Description: {product['Description']}
Rating: {product['Rating']}
"""

    prompt = f"""
You are an e-commerce product assistant.

Retrieved Product Context:
{context}

User Question:
{query}

Answer ONLY using the retrieved product context.

Rules:
1. Do not invent information.
2. Do not use outside knowledge.
3. If the requested information is not present,
say: "The information is not available in the product data."
4. Give a concise answer.
"""

    response = ollama.chat(
        model="qwen2.5:0.5b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return product, similarity, response["message"]["content"]

In [37]:
query = "Tell me about Lenovo Laptop 3"

product, similarity, answer = product_question_answer(query)

print("Retrieved Product:", product["ProductName"])
print("Similarity:", round(similarity, 3))

print("\nAssistant:")
print(answer)

Retrieved Product: Lenovo Laptop 1
Similarity: 0.895

Assistant:
Lenovo Laptop 3 is a popular laptop designed for everyday use with quality performance, suitable for various applications such as online browsing and productivity tasks. The Lenovo brand provides reliable devices suitable for users seeking efficient computing solutions.


In [38]:
query = "What is its price?"

product, similarity, answer = product_question_answer(query)

print("Assistant:")
print(answer)

Assistant:
₹30,316


In [42]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

print("Sentiment analysis model loaded successfully!")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 5487.65it/s]


Sentiment analysis model loaded successfully!


In [43]:
print(reviews["ProductName"].value_counts().head(10))

ProductName
Lenovo Laptop 1    10
Lenovo Laptop 2    10
Lenovo Laptop 3    10
Lenovo Laptop 4    10
Lenovo Laptop 5    10
HP Laptop 1        10
HP Laptop 2        10
HP Laptop 3        10
HP Laptop 4        10
HP Laptop 5        10
Name: count, dtype: int64


In [44]:
def get_product_reviews(product_name):

    product_reviews = reviews[
        reviews["ProductName"]
        .fillna("")
        .str.lower()
        == product_name.lower()
    ]

    return product_reviews

In [46]:
print(reviews.columns.tolist())

['ReviewID', 'ProductID', 'ProductName', 'Review', 'Rating']


In [47]:
print(reviews["ProductName"].value_counts().head(10))

ProductName
Lenovo Laptop 1    10
Lenovo Laptop 2    10
Lenovo Laptop 3    10
Lenovo Laptop 4    10
Lenovo Laptop 5    10
HP Laptop 1        10
HP Laptop 2        10
HP Laptop 3        10
HP Laptop 4        10
HP Laptop 5        10
Name: count, dtype: int64


In [48]:
product_name = reviews["ProductName"].dropna().iloc[0]

print("Selected Product:", product_name)

Selected Product: Lenovo Laptop 1


In [49]:
def get_product_reviews(product_name):

    product_reviews = reviews[
        reviews["ProductName"]
        .fillna("")
        .str.lower()
        == product_name.lower()
    ]

    return product_reviews

In [50]:
product_reviews = get_product_reviews(product_name)

print("Product:", product_name)
print("Number of reviews:", len(product_reviews))

display(product_reviews.head())

Product: Lenovo Laptop 1
Number of reviews: 10


,ReviewID,ProductID,ProductName,Review,Rating
0,1,1,Lenovo Laptop 1,Not worth the price,3
1,2,1,Lenovo Laptop 1,Works perfectly,5
2,3,1,Lenovo Laptop 1,Very good quality,4
3,4,1,Lenovo Laptop 1,Excellent product,5
4,5,1,Lenovo Laptop 1,Very good quality,5


In [51]:
def analyze_sentiments(product_name):

    product_reviews = get_product_reviews(product_name)

    results = []

    for _, row in product_reviews.iterrows():

        review_text = str(row["Review"])

        result = sentiment_model(review_text)[0]

        results.append({
            "Review": review_text,
            "Sentiment": result["label"],
            "Confidence": result["score"]
        })

    return pd.DataFrame(results)

In [52]:
sentiment_results = analyze_sentiments(product_name)

display(sentiment_results.head(10))

,Review,Sentiment,Confidence
0,Not worth the price,NEGATIVE,0.999801
1,Works perfectly,POSITIVE,0.999851
2,Very good quality,POSITIVE,0.999871
3,Excellent product,POSITIVE,0.999862
4,Very good quality,POSITIVE,0.999871
5,Average quality,POSITIVE,0.965560
6,Worth the money,POSITIVE,0.999816
7,Very good quality,POSITIVE,0.999871
8,Highly recommended,POSITIVE,0.999829
9,Poor battery,NEGATIVE,0.999789


In [53]:
def add_neutral_category(df, neutral_threshold=0.60):

    df = df.copy()

    df.loc[
        df["Confidence"] < neutral_threshold,
        "Sentiment"
    ] = "NEUTRAL"

    return df

In [54]:
sentiment_results = add_neutral_category(
    sentiment_results
)

print(
    sentiment_results["Sentiment"].value_counts()
)

Sentiment
POSITIVE    8
NEGATIVE    2
Name: count, dtype: int64


In [55]:
for _, row in sentiment_results.head(10).iterrows():

    print("Review:", row["Review"])
    print("Sentiment:", row["Sentiment"])
    print("Confidence:", round(row["Confidence"], 3))
    print("-" * 50)

Review: Not worth the price
Sentiment: NEGATIVE
Confidence: 1.0
--------------------------------------------------
Review: Works perfectly
Sentiment: POSITIVE
Confidence: 1.0
--------------------------------------------------
Review: Very good quality
Sentiment: POSITIVE
Confidence: 1.0
--------------------------------------------------
Review: Excellent product
Sentiment: POSITIVE
Confidence: 1.0
--------------------------------------------------
Review: Very good quality
Sentiment: POSITIVE
Confidence: 1.0
--------------------------------------------------
Review: Average quality
Sentiment: POSITIVE
Confidence: 0.966
--------------------------------------------------
Review: Worth the money
Sentiment: POSITIVE
Confidence: 1.0
--------------------------------------------------
Review: Very good quality
Sentiment: POSITIVE
Confidence: 1.0
--------------------------------------------------
Review: Highly recommended
Sentiment: POSITIVE
Confidence: 1.0
-----------------------------------

In [56]:
def generate_review_summary(product_name, sentiment_results):

    sample_reviews = sentiment_results.head(20)

    context = ""

    for _, row in sample_reviews.iterrows():

        context += f"""
Review: {row['Review']}
Sentiment: {row['Sentiment']}
"""

    counts = sentiment_results["Sentiment"].value_counts()

    prompt = f"""
You are an e-commerce review analyst.

Product:
{product_name}

Sentiment counts:
Positive: {counts.get('POSITIVE', 0)}
Neutral: {counts.get('NEUTRAL', 0)}
Negative: {counts.get('NEGATIVE', 0)}

Customer reviews:
{context}

Generate:

Overall Opinion
Strengths
Weaknesses
Buying Suggestion

Rules:
- Use only the information provided.
- Do not invent product features.
- Keep the answer concise.
"""

    response = ollama.chat(
        model="qwen2.5:0.5b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [57]:
summary = generate_review_summary(
    product_name,
    sentiment_results
)

print("Product:", product_name)

print("\nSentiment Counts:")
print(
    sentiment_results["Sentiment"].value_counts()
)

print("\nGPT/Ollama Summary:")
print(summary)

Product: Lenovo Laptop 1

Sentiment Counts:
Sentiment
POSITIVE    8
NEGATIVE    2
Name: count, dtype: int64

GPT/Ollama Summary:
**Overall Opinion:** The Lenovo Laptop 1 is a highly recommended product that meets or exceeds customer expectations, offering excellent quality and good value for money.

**Strengths:** The laptop's price makes it an affordable option. The positive reviews indicate it performs well and is well-supported by its user community.

**Weaknesses:** Despite the positive review, there are some potential issues with battery life. While not a major flaw, users often find it disappointing when their devices fail to keep up with their regular usage patterns.

**Buying Suggestion:** For those seeking reliable technology at an affordable price, the Lenovo Laptop 1 is an excellent choice. If you are looking for quality and performance without breaking the bank, consider this review as your guide.


In [58]:
def detect_intent(query):

    prompt = f"""
You are an intent classifier for an e-commerce chatbot.

Classify the user query into exactly ONE of these categories:

FAQ
RECOMMENDATION
PRODUCT
REVIEW

FAQ:
Questions about orders, tracking, cancellation,
returns, delivery, payment or customer support.

RECOMMENDATION:
The user wants a product recommendation.

PRODUCT:
The user asks about a specific product.

REVIEW:
The user asks about customer reviews, opinions,
ratings or sentiment.

User query:
{query}

Return ONLY the category name.
"""

    response = ollama.chat(
        model="qwen2.5:0.5b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    intent = response["message"]["content"].strip().upper()

    valid_intents = [
        "FAQ",
        "RECOMMENDATION",
        "PRODUCT",
        "REVIEW"
    ]

    for category in valid_intents:
        if category in intent:
            return category

    return "PRODUCT"

In [59]:
test_queries = [
    "Where is my parcel?",
    "Recommend a gaming laptop",
    "Tell me about Lenovo Laptop 3",
    "What do customers think about Lenovo Laptop 3?"
]

for query in test_queries:

    print("Question:", query)
    print("Detected Intent:", detect_intent(query))
    print("-" * 60)

Question: Where is my parcel?
Detected Intent: FAQ
------------------------------------------------------------
Question: Recommend a gaming laptop
Detected Intent: PRODUCT
------------------------------------------------------------
Question: Tell me about Lenovo Laptop 3
Detected Intent: PRODUCT
------------------------------------------------------------
Question: What do customers think about Lenovo Laptop 3?
Detected Intent: RECOMMENDATION
------------------------------------------------------------


In [60]:
def ecommerce_chatbot():

    print("=" * 60)
    print("          AI E-COMMERCE CHATBOT")
    print("=" * 60)
    print("Ask your question.")
    print("Type 'exit' to stop.")
    
    while True:

        query = input("\nYou: ").strip()

        if query.lower() in ["exit", "quit"]:
            print("\nChatbot: Goodbye!")
            break

        if not query:
            print("Please enter a question.")
            continue

        # -------------------------
        # INTENT DETECTION
        # -------------------------

        intent = detect_intent(query)

        print("Detected Intent:", intent)

        # -------------------------
        # FAQ
        # -------------------------

        if intent == "FAQ":

            faq_question, faq_answer, score = (
                search_faq_with_threshold(query)
            )

            if faq_question:

                answer = generate_faq_response(
                    query,
                    faq_question,
                    faq_answer
                )

                print("\nAssistant:")
                print(answer)

            else:

                print(
                    "\nAssistant: "
                    "I couldn't find a relevant FAQ."
                )

        # -------------------------
        # RECOMMENDATION
        # -------------------------

        elif intent == "RECOMMENDATION":

            results = retrieve_products(query)

            print("\nTop Retrieved Products:")

            display(
                results[
                    [
                        "ProductName",
                        "Brand",
                        "Price",
                        "Rating",
                        "Similarity"
                    ]
                ]
            )

            answer = generate_product_recommendation(
                query,
                results
            )

            print("\nAssistant:")
            print(answer)

        # -------------------------
        # PRODUCT
        # -------------------------

        elif intent == "PRODUCT":

            product, similarity, answer = (
                product_question_answer(query)
            )

            print("\nRetrieved Product:")
            print(product["ProductName"])

            print("Similarity:",
                  round(similarity, 3))

            print("\nAssistant:")
            print(answer)

        # -------------------------
        # REVIEW
        # -------------------------

        elif intent == "REVIEW":

            product, similarity = retrieve_product(query)

            product_name = product["ProductName"]

            sentiment_results = analyze_sentiments(
                product_name
            )

            if len(sentiment_results) == 0:

                print(
                    "\nAssistant: "
                    "No reviews found for this product."
                )

            else:

                sentiment_results = (
                    add_neutral_category(
                        sentiment_results
                    )
                )

                answer = generate_review_summary(
                    product_name,
                    sentiment_results
                )

                print("\nProduct:", product_name)

                print("\nSentiment Counts:")
                print(
                    sentiment_results[
                        "Sentiment"
                    ].value_counts()
                )

                print("\nAssistant:")
                print(answer)

In [61]:
ecommerce_chatbot()

          AI E-COMMERCE CHATBOT
Ask your question.
Type 'exit' to stop.



You:  Where is my parcel?


Detected Intent: FAQ
Best Matching FAQ: Where is my parcel?
Similarity Score: 1.0

Assistant:
Click on the 'Track Order' section on our platform. You can see where your parcel has been received and when it is expected to arrive. If you need assistance with tracking your order, feel free to ask!



You:  Recommend a gaming laptop


Detected Intent: PRODUCT

Retrieved Product:
Lenovo Laptop 1
Similarity: 0.47

Assistant:
Based on the retrieved product context, a gaming laptop would be ideal for Lenovo users who prioritize performance and multitasking capabilities. A highly rated 5-star recommendation for Lenovo's laptops could be:

Lenovo Gaming Laptop

The Lenovo Gaming Laptop is designed to deliver high-quality gaming experiences, thanks to its high-resolution displays and efficient cooling systems. It also features an all-in-one design for a comfortable user interface.

Price: ₹30,316
Category: Laptop
Rating: 5/5


...

Detected Intent: RECOMMENDATION

Top Retrieved Products:


,ProductName,Brand,Price,Rating,Similarity
3,Lenovo Laptop 4,Lenovo,51158,3.8,0.895198
1,Lenovo Laptop 2,Lenovo,72664,3.8,0.895198
2,Lenovo Laptop 3,Lenovo,7081,4.3,0.895198



Assistant:
Lenovo Laptop 3 by Lenovo is the best recommendation among the options given. It has a rating of 4.3, which is one of the highest possible ratings among the other three products with a rating of 3.8. This indicates that customers generally think highly of this product.

- Brand: Lenovo
- Category: Laptop
- Price: ₹7081
- Description: Laptop by Lenovo suitable for everyday use with quality performance.
- Rating: 4.3



You:  exit



Chatbot: Goodbye!
